<div style="padding: 20px; background: linear-gradient(90deg, #03887dff 0%, #38ef7d 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🔄 Module 8.3: LangGraph Fundamentals for RAG</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Modeling RAG as a cyclic State Machine.</p>
</div>

---

## 1. Why LangGraph?

Standard LangChain is designed for **DAGs** (Directed Acyclic Graphs). This means data flows in one direction: `A → B → C`.
But Agentic RAG requires **Cycles** (Loops). For example: `Retrieve → Grade → (If Bad) → Rewrite → Retrieve Again`.

LangGraph models RAG as a true state machine:
- **State**: A `TypedDict` carrying all intermediate data.
- **Nodes**: Python functions that transform the State.
- **Edges**: Transitions between nodes (fixed or conditional).

In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

# 1. Define the Global State
class RAGState(TypedDict):
    question: str
    documents: List[Document]
    answer: str
    grade: str
    iterations: int

docs = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs."),
    Document(page_content="In LangGraph, nodes are Python functions and edges define control flow."),
    Document(page_content="LangGraph supports cycles and conditional branching for complex agentic workflows."),
    Document(page_content="TypedDict is used to define the state schema in LangGraph applications."),
]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = Chroma.from_documents(docs, embeddings, collection_name="lg_demo_v2")
print("LangGraph Documentation DB Loaded.")

E:\001_Github_Repo_all\Advanced-RAG-Systems\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LangGraph Documentation DB Loaded.


## 2. Defining the Nodes & Routing
We will build a graph that attempts to answer the user. If the retrieved documents aren't relevant, it will rewrite the query and try again, looping until it succeeds or hits an iteration limit.

In [2]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    # --- NODES ---
    def retrieve_node(state: RAGState) -> RAGState:
        print(f"[Node] Retrieving for query: '{state['question']}'")
        docs = vs.similarity_search(state["question"], k=2)
        return {**state, "documents": docs, "iterations": state.get("iterations", 0) + 1}
        
    def grade_node(state: RAGState) -> RAGState:
        print("[Node] Grading retrieved documents...")
        prompt = ChatPromptTemplate.from_template(
            "Is this document directly relevant to '{question}'? Answer strictly 'yes' or 'no'.\nDocument: {doc}"
        )
        grades = []
        chain = prompt | llm | StrOutputParser()
        for doc in state["documents"]:
            r = chain.invoke({"question": state["question"], "doc": doc.page_content})
            grades.append("yes" in r.lower())
            
        grade = "relevant" if any(grades) else "irrelevant"
        print(f"  → Grade: {grade.upper()}")
        return {**state, "grade": grade}
        
    def generate_node(state: RAGState) -> RAGState:
        print("[Node] Generating Final Answer...")
        ctx = "\n".join(d.page_content for d in state["documents"])
        prompt = ChatPromptTemplate.from_template("Answer using context:\n{context}\n\nQuestion: {question}")
        answer = (prompt | llm | StrOutputParser()).invoke({"context": ctx, "question": state["question"]})
        return {**state, "answer": answer}
        
    def rewrite_node(state: RAGState) -> RAGState:
        print("[Node] Rewriting query due to irrelevant docs...")
        prompt = ChatPromptTemplate.from_template("Rewrite this query for better database retrieval. Make it simple: {question}")
        new_q = (prompt | llm | StrOutputParser()).invoke({"question": state["question"]})
        return {**state, "question": new_q.strip()}
        
    # --- CONDITIONAL ROUTER ---
    def route_grade(state: RAGState) -> str:
        # If relevant, move to generate. If we looped 3 times, give up and generate anyway.
        if state["grade"] == "relevant" or state["iterations"] >= 3:
            return "generate"
        return "rewrite"
        
    # --- COMPILE GRAPH ---
    builder = StateGraph(RAGState)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("grade", grade_node)
    builder.add_node("generate", generate_node)
    builder.add_node("rewrite", rewrite_node)
    
    builder.set_entry_point("retrieve")
    builder.add_edge("retrieve", "grade")
    builder.add_conditional_edges("grade", route_grade, {"generate": "generate", "rewrite": "rewrite"})
    builder.add_edge("rewrite", "retrieve") # The Loop Back!
    builder.add_edge("generate", END)
    
    rag_graph = builder.compile()
    
    print("\n--- RUNNING STATE MACHINE ---")
    # This query uses weird vocabulary, triggering a rewrite loop!
    result = rag_graph.invoke({
        "question": "How doth this module oversee intricate computational pipelines?",
        "documents": [], "answer": "", "grade": "", "iterations": 0
    })
    
    print("\n✨ FINAL OUTPUT ✨")
    print(f"Answer: {result['answer']}")
    print(f"Total Loop Iterations: {result['iterations']}")
else:
    print("GROQ_API_KEY missing.")


--- RUNNING STATE MACHINE ---


[Node] Retrieving for query: 'How doth this module oversee intricate computational pipelines?'
[Node] Grading retrieved documents...


  → Grade: IRRELEVANT
[Node] Rewriting query due to irrelevant docs...


[Node] Retrieving for query: '**Improving Database Retrieval for Computational Pipelines**

To optimize database retrieval for intricate computational pipelines, consider the following query:

```sql
SELECT *
FROM computational_pipelines
WHERE status = 'active' AND type = 'intricate';
```

This query retrieves all active intricate computational pipelines from the database. The `status` and `type` conditions help filter the results, reducing the amount of data to be processed.

**Additional Considerations:**

1. **Indexing**: Ensure that the `status` and `type` columns are indexed to improve query performance.
2. **Data Normalization**: Consider normalizing the database schema to reduce data redundancy and improve data integrity.
3. **Data Partitioning**: If the database contains a large number of computational pipelines, consider partitioning the data to improve query performance.

**Example Use Case:**

Suppose you have a database containing various computational pipelines, and you wa

  → Grade: IRRELEVANT
[Node] Rewriting query due to irrelevant docs...


[Node] Retrieving for query: '**Optimized Database Retrieval for Computational Pipelines**

To optimize database retrieval for intricate computational pipelines, consider the following query:

```sql
SELECT id, name, status, type, created_at, updated_at
FROM computational_pipelines
WHERE status = 'active' AND type = 'intricate';
```

**Explanation:**

* Instead of retrieving all columns (`SELECT *`), we only select the necessary columns (`id`, `name`, `status`, `type`, `created_at`, `updated_at`) to reduce data transfer and processing.
* The rest of the query remains the same, filtering the results to include only active intricate pipelines.

**Additional Considerations:**

1. **Indexing**: Ensure that the `status` and `type` columns are indexed to improve query performance.
2. **Data Normalization**: Consider normalizing the database schema to reduce data redundancy and improve data integrity.
3. **Data Partitioning**: If the database contains a large number of computational pipelines

  → Grade: IRRELEVANT
[Node] Generating Final Answer...



✨ FINAL OUTPUT ✨
Answer: Based on the context provided, it appears that the question is asking about optimizing database retrieval for computational pipelines. The provided query and explanation suggest that the goal is to improve query performance by reducing data transfer and processing, as well as improving data integrity and query performance.

To answer the question, I would suggest the following:

**Optimized Database Retrieval for Computational Pipelines**

To optimize database retrieval for intricate computational pipelines, consider the following query:

```sql
SELECT id, name, status, type, created_at, updated_at
FROM computational_pipelines
WHERE status = 'active' AND type = 'intricate';
```

This query retrieves only the necessary columns from the `computational_pipelines` table, reducing data transfer and processing. The rest of the query remains the same, filtering the results to include only active intricate pipelines.

**Additional Considerations:**

1. **Indexing**: E